# Workshop Notebook: Predicting Cytotoxicity of Metal Complexes

This interactive notebook walks you through an end‑to‑end workflow for **cleaning** and **exploring** a dataset of metal complexes tested for cytotoxicity (IC50), which we will use later on to construct machine learning models to predict anti-cancer properties.

You'll alternate between:
- **Read & run** sections (provided code),
- **Your turn** exercises (you write code),
- **Checks** (simple asserts so you know you're on track),
- Optional **challenge** extensions.

Let's start by downloading all the necessary files.


In [ ]:
! wget https://raw.githubusercontent.com/chimie-paristech-CTM/PSL_notebooks/main/cytotoxicity_metal_complexes_application/lib.zip
! wget https://raw.githubusercontent.com/chimie-paristech-CTM/PSL_notebooks/main/cytotoxicity_metal_complexes_application/ruthenium_complexes_dataset.csv

!unzip lib.zip -d lib

# Preliminary description of the dataset

Each datapoint in this dataset corresponds to one experimental measurement of cytotoxicity (IC50) for a ruthenium complex tested against a specific cancer cell line under defined experimental conditions.

**Chemical scope of the dataset**

We exclusively focused on ruthenium(II) complexes with three bidentate ligands.
This means that every complex in the dataset consists of:

- a single Ru(II) metal center, and

- three chelating (bidentate) ligands, which together fully define the coordination environment.

Because the metal center is fixed across the entire dataset, all chemical variability arises from the identity and combination of the ligands.

**Representation choice: ligands only**

In line with previous work, we consistently represent each complex exclusively based on its ligands.

This choice is motivated by two key considerations:

- **Practical considerations**
Constructing valid and consistent SMILES representations for metal complexes is notoriously challenging due to coordination bonds, variable valence descriptions, and limited cheminformatics support. By working at the ligand level, we avoid these issues entirely.

- **Scientific justification**
Since the identity of the coordinating metal (Ru(II)) does not vary within the dataset, encoding the metal explicitly would not add discriminative information. Instead, the ligands are the primary drivers of chemical diversity and biological activity.

As a result, each datapoint is described by:

- the set of three ligands defining the complex, and

- associated experimental metadata (cell line, incubation time, localization, etc.),

- together with a measured IC50 value.

Later in the notebook, we will explore how this ligand-based representation is constructed, cleaned, and analyzed before being used for predictive modeling.

In [ ]:
# ============================================================
# Colab setup (recommended): RDKit via apt + pip extras
# ============================================================
import sys, subprocess, importlib

def run(cmd):
    subprocess.check_call(cmd)

def pip_install(pkgs):
    run([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

print("🔧 Installing RDKit (via apt)…")
run(["bash", "-lc", "apt-get -qq update"])
run(["bash", "-lc", "apt-get -qq install -y python3-rdkit rdkit-data"])
print("✅ RDKit installed (apt).")

print("📦 Installing Python packages…")
pip_install(["pip", "setuptools", "wheel"])
pip_install(["mols2grid>=2.0", "ipywidgets", "tqdm", "scikit-learn", "seaborn", "pandas", "numpy", "matplotlib", "scipy"])

# Enable widgets in Colab
from google.colab import output
output.enable_custom_widget_manager()

# Quick sanity check (no big prints)
try:
    import rdkit
    from rdkit import Chem
    _ = Chem.MolFromSmiles("c1ccccc1")
    print("🎉 Environment ready (RDKit import OK).")
except Exception as e:
    raise RuntimeError("RDKit installed but failed to import. Try 'Runtime > Restart runtime' and rerun this cell.") from e


In [ ]:
# ============================================================
# Data files check (Colab)
# ============================================================
# The workshop expects:
#   - ruthenium_complexes_dataset.csv
#   - lib/  (with any helper utilities used by the notebook)
#
# If you're running on Colab and these files are missing, you can:
#   1) Upload them via the file picker, or
#   2) Mount Google Drive and point to the folder.
# ============================================================

import os, pathlib, sys

expected_csv = "ruthenium_complexes_dataset.csv"
expected_lib = "lib"

cwd = pathlib.Path(".").resolve()
print("📁 Current working directory:", cwd)

missing = []
if not pathlib.Path(expected_csv).exists():
    missing.append(expected_csv)
if not pathlib.Path(expected_lib).exists():
    missing.append(expected_lib + "/")

if not missing:
    print("✅ Found expected workshop files.")
else:
    print("⚠️ Missing:", ", ".join(missing))
    try:
        from google.colab import files
        print("\n➡️ Upload the missing files now (CSV + lib folder contents).")
        print("   Tip: upload the CSV directly; for the lib folder, upload a ZIP and unzip it.")
        # Trigger the upload UI (students can cancel if they mounted Drive instead)
        # Comment this out if you don't want the upload dialog to pop automatically.
        # uploaded = files.upload()
    except Exception:
        print("\nℹ️ If you're not on Colab, place the missing files next to this notebook.")

# Optional helper: if you uploaded a ZIP (e.g., lib.zip), unzip it
for z in ["lib.zip", "workshop_files.zip"]:
    if pathlib.Path(z).exists():
        print(f"📦 Found {z} — unzipping...")
        import zipfile
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall(".")
        print("✅ Unzipped.")


First, we have to load some of the packages we will use throughout this notebook. We will also define a soft assert function to provide warnings throughout this notebook.

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("white")

# Properties of molecules
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole #RDKit drawing 
from rdkit.Chem import rdDepictor # A few settings to improve the quality of structures
IPythonConsole.ipython_useSVG = True
rdDepictor.SetPreferCoordGen(True)
import mols2grid #The mols2grid library provides a convenient way of displaying molecules in a grid

# -------------------------------
# Workshop helper: soft checks
# -------------------------------
def soft_assert(condition, message=""):
    """Soft check: prints a warning instead of stopping the notebook."""
    if condition:
        print(f"✅ {message}".strip())
        return True
    else:
        print(f"⚠️ {message}".strip())
        return False

# Data Preprocessing

We start by importing the dataset. We will drop every entry for which the ligands are not fully defined.

In [ ]:
df_metal_complexes= pd.read_csv("ruthenium_complexes_dataset.csv", dtype={'L1': str, 'L2': str, 'L3': str})
df_metal_complexes.dropna(subset=['L1', 'L2', 'L3'], how='any', inplace=True)
df_metal_complexes.reset_index(drop=True, inplace=True)

## First look at the dataset
Before going into further detail, let's take a look at the structure and size of the dataset.

### Your turn: basic dataset counts
Fill in the code below to compute:
- number of rows (entries) and columns
- number of **unique papers** (use the DOI column)

> Hint: `df.shape` and `nunique()` are your friends.


In [ ]:
# SOLUTION: basic counts
df = df_metal_complexes  # alias for convenience

ligand_cols = ["L1", "L2", "L3"]

# 1) How many entries and columns?
n_entries = int(df.shape[0])
n_columns = int(df.shape[1])

# 2) How many individual papers have been analyzed (unique DOI)?
n_unique_papers = int(df['DOI'].nunique(dropna=True))

print(n_entries, n_columns, n_unique_papers)

Let's now also determine how many unique complexes and ligands exist in the dataset. To determine this accurately, we first need to perform some preliminary steps. 

First, we will make all SMILES canonical.

In [ ]:
def get_canonical_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol)

### Your turn: canonicalize all ligand SMILES
Now that `get_canonical_smiles(smiles)` is defined, apply it to **every ligand column**.

**Task:**
- Run canonicalization for `L1`, `L2`, and `L3`.
- Confirm (with a simple check) that there are no missing values in those ligand columns afterwards.

In [ ]:
# SOLUTION: make all ligands canonical
ligand_cols = ['L1', 'L2', 'L3']

for col in ligand_cols:
    df_metal_complexes[col] = df_metal_complexes[col].apply(get_canonical_smiles)

A metal complex is uniquely determined by its ligands. Since the order of the ligands has no chemical significance, we have to represent the complex in a position independent manner. This is done through the procedure below.

In [ ]:
df_metal_complexes["ligands_key"] = df_metal_complexes.apply(
    lambda row: tuple(sorted([row["L1"], row["L2"], row["L3"]])),
    axis=1
)

### ✅ Check: ligand SMILES are canonical and parseable

Run this cell after you canonicalize `L1`, `L2`, and `L3`.
It should complete without errors.

In [ ]:
from rdkit import Chem
import numpy as np

# Basic presence checks
for col in ["L1", "L2", "L3"]:
    soft_assert(col in df_metal_complexes.columns, f"Missing column {col}")
    soft_assert(df_metal_complexes[col].notna().all(), f"NaNs found in {col} after canonicalization.")

# Parseability checks (RDKit can read all ligand SMILES)
for col in ["L1", "L2", "L3"]:
    mols = df_metal_complexes[col].apply(Chem.MolFromSmiles)
    n_bad = int(mols.isna().sum())
    soft_assert(n_bad == 0, f"{n_bad} invalid SMILES found in {col}.")

# Canonicalization idempotence: canonical_smiles shouldn't change an already-canonical column
_tmp = df_metal_complexes[["L1","L2","L3"]].copy()
for col in ["L1","L2","L3"]:
    _tmp[col] = df_metal_complexes[col].apply(get_canonical_smiles)
soft_assert((_tmp["L1"].values == df_metal_complexes["L1"].values).all(), "(_tmp[\"L1\"].values == df_metal_complexes[\"L1\"].values).all()")
soft_assert((_tmp["L2"].values == df_metal_complexes["L2"].values).all(), "(_tmp[\"L2\"].values == df_metal_complexes[\"L2\"].values).all()")
soft_assert((_tmp["L3"].values == df_metal_complexes["L3"].values).all(), "(_tmp[\"L3\"].values == df_metal_complexes[\"L3\"].values).all()")

### Your turn: count the number of complexes and ligands
Now that the SMILES strings have been cleaned up, let's determine:
- number of **unique complexes** (a complex is uniquely determined by its ligand set)
- number of **unique ligands** per ligand column, and **across all ligand columns**

In [ ]:
# 1) How many unique complexes?
n_unique_complexes = int(df_metal_complexes['ligands_key'].nunique())

# 2) How many unique ligands in each ligand column?
n_unique_l1 = int(df[ligand_cols[0]].astype(str).nunique())
n_unique_l2 = int(df[ligand_cols[1]].astype(str).nunique())
n_unique_l3 = int(df[ligand_cols[2]].astype(str).nunique())

# 5) How many unique ligands total across L1/L2/L3?
# Flatten all ligand entries across the three columns into one 1D series
all_ligands = df[ligand_cols].astype(str).stack(dropna=True)
n_unique_ligands_total = int(all_ligands.nunique())

print(n_unique_complexes, n_unique_ligands_total)

# Simple checks (should pass once filled)
soft_assert(isinstance(n_unique_complexes, int) and n_unique_complexes > 0)
soft_assert(isinstance(n_unique_ligands_total, int) and n_unique_ligands_total > 0)


Next, let's turn every ligand in its corresponding mol object.

In [ ]:
df_metal_complexes['mol1'] = df_metal_complexes.L1.apply(lambda x: Chem.MolFromSmiles(x))
df_metal_complexes['mol2'] = df_metal_complexes.L2.apply(lambda x: Chem.MolFromSmiles(x))
df_metal_complexes['mol3'] = df_metal_complexes.L3.apply(lambda x: Chem.MolFromSmiles(x))
# YOUR CODE HERE

Let's now clean the column names (remove any spaces, etc) and convert all IC50s to floats.

In [ ]:
def convert_to_float(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return None

df_metal_complexes.rename(columns={'IC50 (μM)': 'IC50', 'Incubation Time (hours)': 'IncubationTime', 'Partition Coef logP': 'logP', 'Cell Lines ': 'Cells'}, inplace=True)

def convert_to_float(value):
    if value.startswith('>'):
        value_cleaned = value[1:]
    elif value.startswith('<'):
        value_cleaned = value[1:]
    else:
        value_cleaned = value

    try: 
        return float(value_cleaned)
    except(ValueError, TypeError):
        print(value_cleaned)
        return None

df_metal_complexes['IC50'] = df_metal_complexes['IC50'].apply(convert_to_float)

To drop duplicates of a column of lists:

In [ ]:
def drop_duplicates(df, column):
    # Convert lists in specified column to tuples
    df[column] = df[column].apply(tuple)
    
    # Drop duplicate rows based on the values within the tuples in the specified column
    df.drop_duplicates(subset=[column], inplace=True)
    
    # Convert tuples in specified column back to lists
    df[column] = df[column].apply(list)

    print(len(df))

### Your turn: build a ligand grid
We’d like to **visualize ligands** and how often they appear across unique complexes.

**Task:**
1. Sort ligands by frequency (you already have `ligand_occurence_dict`).
2. Build a `mol_grid_dict` with keys:
   - `'SMILES'` (list of SMILES)
   - `'occurences'` (list of counts)
   - `'molecules'` (RDKit Mol objects)
3. Display using `mols2grid.display(...)`.

> Tip: `sorted(ligand_occurence_dict.items(), key=lambda x: x[1], reverse=True)`


In [ ]:
# SOLUTION: create mol_grid_dict then display the grid
from rdkit import Chem

# Count ligand occurrences across all ligand columns
ligand_counts = df_metal_complexes[ligand_cols].stack().value_counts(dropna=True)

# Sorted list of (smiles, count) from most frequent to least
sorted_ligands = list(ligand_counts.items())

smiles_list = [s for s, _ in sorted_ligands]
count_list = [int(c) for _, c in sorted_ligands]
mol_list = [Chem.MolFromSmiles(s) for s in smiles_list]

mol_grid_dict = {
    "SMILES": smiles_list,
    "occurences": count_list,
    "molecules": mol_list,
}

# Display
mols2grid.display(mol_grid_dict, subset=["img", "occurences"], substruct_highlight=True)

### ⭐ Challenge: show the *top-K* ligands only

Modify your ligand grid to display only the **top K** most frequent ligands (e.g. K=20 or 50).

This keeps the visualization readable and makes it faster in Colab.


In [ ]:
# CHALLENGE (optional)
K = 20

# 1) Count ligand frequencies across all ligand columns
ligand_counts = (
    df[ligand_cols]
    .astype(str)
    .stack(dropna=True)
    .value_counts()
)

topK_ligands = ligand_counts.head(K).index.tolist()

# 2) Build a DataFrame for mols2grid
import pandas as pd
grid_df_K = pd.DataFrame({
    "smiles": topK_ligands,
    "count": ligand_counts.head(K).values
})

# 3) Display (mols2grid reads SMILES and renders structures)
import mols2grid
mols2grid.display(
    grid_df_K,
    smiles_col="smiles",
    subset=["smiles", "count"]
)

# Analyze distributions

We are going to start by looking at the IC50 distribution of our entries. By **playing with the parameters p and n**, we can visualize the distribution in more or less detail.

In [ ]:
def generate_bar_plot(df, title, xlabel, ylabel, num_columns, figsize=(5,3)):
    plt.figure(figsize=figsize)
    df.plot(kind='bar', color=sns.color_palette('RdBu', num_columns))
    plt.title(title)
    plt.xticks(fontsize=10, color='k', fontweight='light')
    plt.yticks(fontsize=10, color='k', fontweight='light')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
n = 10  # Number of subsets for the range [0, 100]
p = 4  # Number of subsets for the range [100, 500]

bin_edges_1 = np.linspace(0, 100, n + 1)
bin_edges_2 = np.linspace(100, 500, p + 1)
bin_edges = np.concatenate((bin_edges_1, bin_edges_2[1:]))

# Create bins and labels
bins = list(bin_edges)
labels = [f'{bins[i]}-{bins[i+1]}' for i in range(len(bins)-1)]

# Apply binning and count the number of compounds in each bin
plot_metals = df_metal_complexes.copy()
plot_metals['Y_bins'] = pd.cut(plot_metals['IC50'], bins=bin_edges, labels=labels, include_lowest=True)
counts = plot_metals['Y_bins'].value_counts().sort_index()

# Plot the bar plot
generate_bar_plot(counts, 'Number of entries by IC50 Value Range', 'IC50 Value Range (μM)', 
                  'Number of entries', len(bin_edges))

### ⭐ Challenge: transform IC50 → pIC50 and compare distributions

IC50 values (in μM) are often right-skewed. A common transformation is:

$$
\mathrm{pIC50} = -\log_{10}(\mathrm{IC50~in~M}) = 6 - \log_{10}(\mathrm{IC50~in~\mu M})
$$

**Task**
1. Create a new column `pIC50` from `IC50`.
2. Plot a histogram (or KDE) of `IC50` **and** `pIC50` and compare.
3. Quantify skewness before/after (optional).

> This is optional, but very helpful for understanding why later models behave better when using pIC50.


In [ ]:
# CHALLENGE (optional): create pIC50
import numpy as np

# 1) Create pIC50
# pIC50 = -log10(IC50 [M])
df_metal_complexes["pIC50"] = -np.log10(df_metal_complexes["IC50"])

# 2) Plot distributions (example using seaborn)
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

sns.histplot(df_metal_complexes["IC50"], bins=40, ax=axes[0])
axes[0].set_xscale("log")
axes[0].set_title("IC50 (log scale)")
axes[0].set_xlabel("IC50")

sns.histplot(df_metal_complexes["pIC50"], bins=40, ax=axes[1])
axes[1].set_title("pIC50")
axes[1].set_xlabel("pIC50")

plt.tight_layout()
plt.show()


We now consider the cell line distribution.

In [ ]:
value_counts = df_metal_complexes['Cells'].value_counts()
cells = value_counts[value_counts >= 1]

# Plot the bar plot
generate_bar_plot(cells, 'Number of entries tested for each cell line', 'Cell lines', 'Number of entries', len(value_counts), 
                  (18, 4))

We now consider the incubation time distribution.

In [ ]:
value_counts = df_metal_complexes['IncubationTime'].value_counts()
incubation_times = value_counts[value_counts >= 1]

generate_bar_plot(incubation_times, 'Number of entries for each incubation time', 'Incubation time', 'Number of entries', len(value_counts), 
                  (4, 2))

And finally, we consider the localization distribution.

In [ ]:
value_counts = df_metal_complexes['Localisation'].value_counts()
localisations = value_counts[value_counts >= 1]

generate_bar_plot(localisations, 'Number of entries for each localisation', 'Localisation', 'Number of entries', len(value_counts), 
                  (4, 2))

## Impact analysis of confounding factors (incubation time, localization, etc.)

Here, we will test whether variation in some of the experimental variables have a significant
effect on the measured IC50 values. This analysis can give us an idea about whether these
variables can be safely ignored (because they don't affect the experimental outcome), 
or whether different values for them can result in very different IC50 values — in this case,
we should test whether including these variables in the representation of the datapoints
results in a more accurate predictive model.

### Your turn: pick metadata fields and quantify their impact on the measured IC50 values.

Go through columns among:
- `IncubationTime`
- `Localisation`
- `Cells` (cell line)

Suggested approaches:
- Box plots of IC50 / pIC50 across categories
- Simple ANOVA / Kruskal-Wallis across groups
- Effect size summaries (median differences, IQR, etc.)


### ✅ Check: required metadata columns exist

Before impact analysis, confirm the expected metadata fields are available.


In [ ]:
required_cols = ["IC50", "Cells", "IncubationTime", "Localisation"]
missing = [c for c in required_cols if c not in df_metal_complexes.columns]
soft_assert(not missing, f"Missing required columns: {missing}")
soft_assert(df_metal_complexes["IC50"].notna().all(), "IC50 contains NaNs (did conversion fail?).")

In [ ]:
def generate_box_plot(title, df, x_column, y_column, xlabel, ylabel, fontsize=50, figsize=(80,30)):
    plt.figure(figsize=figsize)
    sns.boxplot(x=x_column, y=y_column, data=df, palette='flare')
    plt.title(title, fontsize=60)
    plt.xlabel(xlabel, fontsize=fontsize)
    plt.ylabel(ylabel, fontsize=fontsize)
    plt.xticks([]) 
    plt.yticks(fontsize=fontsize)
    plt.show()

# for technical plotting reasons, we need to turn the ligands_key into a string
df_metal_complexes["ligands_key_str"] = df_metal_complexes["ligands_key"].apply(
    lambda x: " | ".join(x)
)

We start with the influence of the cell line.

In [ ]:
df_cell_line_analysis = df_metal_complexes.copy()

First, we are going to **only keep compound that have been tested on 2 different cell lines or more.**

In [ ]:
# We count the number of unique values in the 'Cells' column for each 'ligands_key'.
counts = df_cell_line_analysis.groupby('ligands_key_str')['Cells'].nunique()

# We only keep the rows where the 'ligands_key_str' column has more than one unique value in the 'Cells' column.
cell_line_analysis = df_cell_line_analysis[df_cell_line_analysis['ligands_key_str'].isin(counts[counts > 1].index)]

To filter out the potentiall confounding factors 'Incubation Time' and 'Localization', we focus on their most common values.

In [ ]:
cell_line_analysis_48 = cell_line_analysis[cell_line_analysis['IncubationTime'] == 48]
cell_line_analysis_48c = cell_line_analysis_48[cell_line_analysis_48['Localisation'] == 'cytoplasm']

generate_box_plot("IC50 variation depending on the cell line for various complexes", cell_line_analysis_48c, 
                  "ligands_key_str", "IC50", "Individual complexes", "IC50 (μM)")

In [ ]:
cell_line_analysis_48n = cell_line_analysis_48[cell_line_analysis_48['Localisation'] == 'nucleus']

generate_box_plot("IC50 variation depending on the cell line for various complexes", cell_line_analysis_48n, 
                  "ligands_key_str", "IC50", "Individual complexes", "IC50 (μM)")

In [ ]:
cell_line_analysis_4 = cell_line_analysis[cell_line_analysis['IncubationTime'] == 4]
cell_line_analysis_4c = cell_line_analysis_4[cell_line_analysis_4['Localisation'] == 'cytoplasm']

generate_box_plot("IC50 variation depending on the cell line for various complexes", cell_line_analysis_4c, 
                  "ligands_key_str", "IC50", "Individual complexes", "IC50 (μM)")

In [ ]:
cell_line_analysis_4n = cell_line_analysis_4[cell_line_analysis_4['Localisation'] == 'nucleus']

generate_box_plot("IC50 variation depending on the cell line for various complexes", cell_line_analysis_4n, 
                  "ligands_key_str", "IC50", "Individual complexes", "IC50 (μM)")

Clearly, the cell line plays a role in the IC50 values!

Next, we consider the influence of the incubation time.

In [ ]:
df_incubation_analysis = df_metal_complexes.copy()

# To have a hashable object that can be run by "groupby" function, we have to modify the 'ligands_key_str' column
df_incubation_analysis['ligands_key_str'] = df_incubation_analysis['ligands_key_str'].apply(lambda x: ' '.join(str(x)))

# We only keep the rows where the 'ligands_key_str' + 'Cells' tuple has more than one unique value in the 'IncubationTime' column.
df_incubation_analysis = df_incubation_analysis.groupby(['ligands_key_str', 'Cells']).filter(lambda x: x['IncubationTime'].nunique() > 1)

### Checkpoint: unique complexes in incubation-time subset
Compute how many unique complexes are present in `df_incubation_analysis` and store it in `n_unique_complexes_incubation_subset`.


In [ ]:
# SOLUTION
# Count unique complexes in the subset used for incubation-time impact analysis
n_unique_complexes_incubation_subset = int(df_incubation_analysis["ligands_key_str"].nunique())

We are then going to display the different IC50 values obtained for different incubation times for each *compound + cell line* pair, to reveal the influence of incubation time in isolation.

In [ ]:
# Combine columns 'ligands_key_str' and 'Cells' into tuples, so that we have a compound tested on a specific cell line
df_incubation_analysis['R+C'] = df_incubation_analysis.apply(lambda x: ' '.join(str(x['ligands_key_str'])) + x['Cells'], axis=1)

In [ ]:
df_incubation_analysis_c = df_incubation_analysis[df_incubation_analysis['Localisation'] == 'nucleus'] # cytoplasma is empty

generate_box_plot("IC50 variation depending on the incubation time for various complexes", df_incubation_analysis_c, 
                  "R+C", "IC50", "Complexes tested on a specific cell line", "IC50 (μM)")

In [ ]:
df_incubation_analysis_n = df_incubation_analysis[df_incubation_analysis['Localisation'] == 'mitochondria']

generate_box_plot("IC50 variation depending on the incubation time for various complexes", df_incubation_analysis_n, 
                  "R+C", "IC50", "Complexes tested on a specific cell line", "IC50 (μM)")


=> Complexes tried on a specific cell line are only seldomly evaluated at different incubation times, but when they are, we get significant variation of the IC50 values!    

Finally, we consider the effect of the localization.

**We expect the localisation to have no impact on the IC50** : if a compound has been found in various locations in the cell, it still undergoes only one IC50 test on each cell line.

**First, we only keep the compounds that have been found in different localisations :** 

In [ ]:
df_localisation_analysis = df_metal_complexes.copy()
# To have a hashable object that can be runed by "groupby" function, we have to modify the 'ligands_key_str' column
df_localisation_analysis['ligands_key_str'] = df_localisation_analysis['ligands_key_str'].apply(lambda x: ' '.join(str(x)))

# filter out the NaN values for the localisation
df_localisation_analysis = df_localisation_analysis.dropna(subset=['Localisation'])

# We only keep the rows where the 'ligands_key_str' + Cells + IncubationTime' tuple has more than one unique value in the 'Localisation' column.
df_localisation_analysis = df_localisation_analysis.groupby(['ligands_key_str', 'Cells', 'IncubationTime']).filter(lambda x: x['Localisation'].nunique() > 1)

### Checkpoint: unique complexes in localisation subset
Compute how many unique complexes are present in `df_localisation_analysis` and store it in `n_unique_complexes_localisation_subset`.


In [ ]:
# SOLUTION
# Count unique complexes in the subset used for localisation impact analysis
n_unique_complexes_localisation_subset = int(df_localisation_analysis["ligands_key_str"].nunique())

In [ ]:
# Combine columns 'ligands_key_str' and 'Cells', so that we have a compound tested on a specific cell line
df_localisation_analysis['R+C+I'] = df_localisation_analysis.apply(
    lambda x: ' '.join(str(x['ligands_key_str'])) + x['Cells'] + str(x['IncubationTime']), axis=1)

generate_box_plot("IC50 variation depending on the localization", df_localisation_analysis, 
                  "R+C+I", "IC50", "Complexes tested on a specific cell line with a specific incubation time", "IC50 (μM)")

=> The localization indeed appears to have no effect on the IC50 values